# Progetto 1: Canale di depolarizzazione

In questo progetto implementeremo il **canale di depolarizzazione** in Qiskit e lo testeremo con la tomografia di stato su un simulatore e, facoltativamente, su un dispositivo reale.

Il canale di depolarizzazione è uno dei modelli più comuni di decoerenza dei qubit grazie alle sue interessanti proprietà di simmetria. Possiamo descriverlo affermando che, con probabilità $1-p$ il qubit rimane intatto, mentre con probabilità $p$ si verifica un errore. L'errore può essere un errore di inversione di bit, descritto dall'azione di $\sigma_x$, un errore di inversione di fase, descritto dall'azione di $\sigma_z$, o entrambi, descritti dall'azione di $\sigma_y$. La mappa dinamica di un sistema quantistico aperto markoviano soggetto a rumore depolarizzante può essere scritta come

\begin{align}
\Phi_t \rho_S = \left[1-\frac 3 4 p(t)\right] \rho_S + \frac{p(t)}{4} \sum_i \sigma_i \rho_S \sigma_i,
\end{align}

dove $i=x,y,z$ e $p(t)=1 - e^{-\gamma t}$, con $\gamma$ il rate di decadimento di Markov. 

Il canale di depolarizzazione può essere implementato, per qualsiasi valore di $p\equiv p(t) \in [0, 1]$, con il circuito illustrato nella figura sopra. 
Tre qubit ausiliari vengono preparati in uno stato $| \psi_\theta \rangle = \cos \theta/2 | 0 \rangle + \sin \theta/2 | 1 \rangle$
e vengono utilizzati come controlli rispettivamente per una rotazione controllata-$X$ (CNOT), una rotazione controllata-$Y$ e una rotazione controllata-$Z$.
In questo modo, ogni gate verrà applicato con una probabilità di $\sin^2 \theta/2$.

L'angolo di rotazione $\theta$ deve essere scelto in modo tale che ciascuno dei gate venga applicato con probabilità $p$. Si noti che applicare prima $X$ e poi $Y$, senza applicare $Z$, equivale (a parte le fasi globali) ad applicare solo $Z$, e così via. L'equazione risultante che lega $\theta$ a $p$ è quindi

\begin{equation}
    \sin^2 \frac \theta 2 \cos^4\frac\theta2 + \sin^4 \frac\theta 2 \cos^2 \frac \theta 2 = \frac p 4,
\end{equation}
con soluzione $\theta(p) = \frac 12 \arccos(1 - 2 p)$.

In [1]:
####################################
#       Depolarizing channel       #
####################################

from qiskit import QuantumRegister, QuantumCircuit
import numpy as np

# Quantum register
q = QuantumRegister(4, name="q")

# Quantum circuit
depolarizing = QuantumCircuit(q)

# Depolarizing channel acting on q_0
## Qubit identification
system = 0
a_0 = 1
a_1 = 2
a_2 = 3

## Define rotation angle
theta = np.pi/4

## Construct circuit
depolarizing.ry(theta, q[a_0])
depolarizing.ry(theta, q[a_1])
depolarizing.ry(theta, q[a_2])
depolarizing.cx(q[a_0], q[system])
depolarizing.cy(q[a_1], q[system])
depolarizing.cz(q[a_2], q[system])

# Draw circuit
depolarizing.draw(output='mpl')

ModuleNotFoundError: No module named 'qiskit'

## Attività 1 (1 punto)

Crea una funzione che dia come risultato un circuito quantistico che implementi un canale di depolarizzazione con parametro $p$ su uno specifico `system` di qubit, usando tre qubit ausiliari `ancillae = [a1, a2, a3]`.

In [ ]:
def depolarizing_channel(q, p, system, ancillae):
    """Returns a QuantumCircuit implementing depolarizing channel on q[system]
    
    Args:
        q (QuantumRegister): the register to use for the circuit
        p (float): the probability for the channel between 0 and 1
        system (int): index of the system qubit
        ancillae (list): list of indices for the ancillary qubits
        
    Returns:
        A QuantumCircuit object
    """
    
    # Write the code here...

## Attività 2 (1 punto)
Scrivi uno stato iniziale (`initial_state`) per un circuito che prepari il `system` qubit in uno stato iniziale con popolazioni e coerenze non nulle (parti reali e immaginarie)

In [ ]:
# Let's fix the quantum register and the qubit assignments

# We create the quantum circuit
q = QuantumRegister(4, name='q')

# Index of the system qubit
system = 1

# Indices of the ancillary qubits
ancillae = [0, 2, 3]

## Attività 3 (4 punti)
Preparare l'esperimento di tomografia di stato con cui analizzeremo lo stato generato dal circuito e dal rumore depolarizzante. In questo caso, ricostruiremo le matrici di densità e le [fidelità di stato](https://en.wikipedia.org/wiki/Fidelity_of_quantum_states).

1. Per diversi valori di $p \in [0, 1]$:

    1. Combina `initial_state` e `depolarizing_channel` in un circuito.

    2. Prepara ed esegui `StateTomography` da `qiskit_experiments.library.tomography` per eseguire la tomografia **solo** sul qubit di sistema utilizzando un simulatore.

      - Calcola anche la fedeltà rispetto allo **stato del qubit di sistema** in `initial_state` (senza il canale di depolarizzazione). Potrebbe essere necessario [`partial_trace`](https://qiskit.org/documentation/apidoc/quantum_info.html#qiskit.quantum_info.partial_trace) da `qiskit.quantum_info`.

      - Calcolare le barre di errore per le fedeltà con il [bootstrapping](https://en.wikipedia.org/wiki/Bootstrapping_(statistics)). A tal fine, utilizzare `StateTomographyAnalysis` da `qiskit_experiments.library.tomography` con alcuni argomenti e fornirlo a `StateTomography`.

    3. Raccogli le matrici di densità, le fedeltà e gli errori delle fedeltà.
<br/><br/>
2. Descrivi brevemente con parole tue cosa fa `StateTomography`. Cosa bisogna misurare? Come ci assicuriamo che la matrice di densità sia fisica? ([Tomografia dello stato quantistico](https://qiskit.org/ecosystem/experiments/manuals/verification/state_tomography.html) nella documentazione di `qiskit-experiments` e [Sistemi quantistici aperti con Qiskit](https://matteoacrossi.github.io/oqs-jupyterbook/preliminaries.html) potrebbero essere d'aiuto. )

In [ ]:
# For example, let's consider 10 equally spaced values of p
import numpy as np
p_values = np.linspace(0, 1, 10)

## Attività 4 (4 punti)
1. Calcola numericamente la matrice di densità esatta del qubit del sistema dopo il canale di depolarizzazione in funzione di $p$.
2. Traccia i valori di $\rho_{00}$, $\rho_{11}$, $\mathrm{Re}(\rho_{01})$, $\mathrm{Im}(\rho_{01})$ in funzione di $p$ e confrontali con la previsione analitica.
3. Trova numericamente le fedeltà esatte del qubit del sistema dopo il canale di depolarizzazione in funzione di $p$.
4. Traccia sia le fedeltà esatte che quelle simulate del qubit del sistema rispetto allo stato iniziale del sistema in funzione di $p$. Aggiungi ai grafici le barre di errore calcolate nell'attività 3.

Fatti salvi gli errori statistici dovuti al numero finito di shot, i punti simulati dovrebbero essere vicini alla previsione analitica. Per la fedeltà, le barre di errore coprono 1 deviazione standard (~68%).

In [ ]:
import matplotlib.pyplot as plt

## Attività facoltativa
Esegui tutte le attività su un dispositivo reale con mitigazione del rumore e confronta i risultati con quelli della simulazione. A tal fine puoi utilizzare [`MitigatedStateTomography`](https://qiskit.org/ecosystem/experiments/stubs/qiskit_experiments.library.tomography.MitigatedStateTomography.html#qiskit_experiments.library.tomography. MitigatedStateTomography) invece di `StateTomography`.

Si noti che `MitigatedTomographyAnalysis` funziona in modo leggermente diverso da `StateTomographyAnalysis`. In caso di problemi, provare a specificare la classe di analisi al momento dell'esecuzione dell'esperimento di tomografia, anziché durante l'inizializzazione.

Inoltre, è possibile ottenere il backend del dispositivo reale utilizzando il codice seguente (le primitive come `Sampler` sono relativamente nuove, quindi non sono ancora supportate in `qiskit-experiments`). Infine, assicurati di aver creato e salvato il tuo account [IBM Quantum](https://quantum-computing.ibm.com/)!

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
# Replace ibm_brisbane with the backend you want to run this on  
backend = service.get_backend("ibm_brisbane")